# M3 ext. — Validación walk-forward

Análisis de **robustez**, no un sustituto del experimento principal. El split
único responde a «¿funcionó en 2025?»; el walk-forward responde a
«¿**funciona sistemáticamente**, o 2025 fue un año particular?».

La serie se recorre por bloques anuales sucesivos: en cada pliegue el modelo se
reentrena con todo lo anterior al bloque y se evalúa solo sobre ese bloque, que
nunca ha visto. El resultado no es una métrica sino una **secuencia** de
métricas a lo largo de diez años.

Corresponde a la Sección *Validación walk-forward* del Capítulo 4 y a la
sección de robustez del Capítulo 5.

Como el resto de cuadernos, este es un **orquestador fino**: toda la lógica
vive en `tfm.walkforward`, `tfm.validation` y `tfm.evaluation`.


## 1. El diseño de pliegues, antes de entrenar nada

Conviene mirar los pliegues antes de gastar horas de CPU. Dos cosas que
comprobar aquí:

- **`train_end < test_start` siempre**, con la distancia de la purga.
- **El número de pliegues varía por activo.** BTC-USD (histórico desde 2014) y
  ETH-USD (desde 2017) tienen menos que los índices. Es correcto y se reporta
  en la columna `n_folds`; la comparación entre activos se hace con
  **proporciones**, nunca con recuentos.


In [ ]:
import pandas as pd

from tfm import config, data, features, validation, walkforward

raw = data.load_raw()
ticker = "^GSPC"  # change or loop over config.TICKERS

dataset = features.build_dataset(raw[ticker])
folds = validation.make_walkforward_folds(dataset.index)
validation.folds_to_frame(ticker, folds)


In [ ]:
# Folds per asset: fewer when the history is short
pd.DataFrame(
    {
        "n_folds": len(validation.make_walkforward_folds(
            features.build_dataset(df).index)),
        "history_from": df.index.min().date(),
    }
    for t, df in raw.items()
).assign(ticker=list(raw)).set_index("ticker")


## 2. La purga en la frontera

La variable objetivo es la dirección a `N` sesiones vista, así que `y[t]` se
construye con el precio en `t+N`. Las **últimas N filas de cada ventana de
entrenamiento** tienen, por tanto, una etiqueta fabricada con datos que ya
pertenecen al bloque de prueba: entrenar con ellas es fuga de información.

La purga las elimina. Con `N=1` es una sola fila y el efecto numérico es
despreciable, pero en el brazo de sensibilidad `N=5` sería material.


In [ ]:
n = config.TARGET_HORIZON_N
for fold in folds[:3]:
    last_train = dataset.index.get_loc(fold.train_end)
    first_test = dataset.index.get_loc(fold.test_start)
    print(f"{fold.test_year}: train_end={fold.train_end.date()} "
          f"test_start={fold.test_start.date()} "
          f"gap={first_test - last_train} filas "
          f"(purga={fold.purge_rows}) "
          f"| etiqueta de la ultima fila de train resuelve en "
          f"{dataset.index[last_train + n].date()}")


## 3. Prueba de humo

Un activo y dos modelos, con los hiperparámetros congelados. Sirve para
verificar los pliegues y las columnas de salida **antes** de escalar.

`retune=False` es una comodidad de depuración, no una opción neutra: la rejilla
se eligió con datos hasta 2024, así que un pliegue de 2016 estaría usando una
configuración escogida conociendo su propio futuro. Nunca para la tanda que se
reporta.


In [ ]:
specs = walkforward.build_model_specs()
smoke = walkforward.run_walkforward_asset(
    ticker,
    dataset,
    {k: v for k, v in specs.items() if k in ("naive", "logreg")},
    retune=False,
    frozen_params=walkforward.load_frozen_params(),
    folds=folds,
)
smoke[["model", "test_year", "n_test", "base_rate", "accuracy",
       "balanced_accuracy", "above_baseline", "p_value_binomial"]]


## 4. Tanda completa

**Aviso de coste.** Con reentrenamiento por pliegue son del orden de **2–3 h**
para los catorce activos y los seis modelos. Desde la línea de comandos:

```powershell
python -m tfm.pipeline --offline --walk-forward --wf-backtest --lstm
```

La celda siguiente hace lo mismo desde el cuaderno. Está en modo reducido a
propósito: quita `tickers` y `model_names` para la tanda completa.


In [ ]:
result = walkforward.run_walkforward(
    tickers=["^GSPC", "^IBEX"],          # omit to run all 14 assets
    model_names=["naive", "logreg"],     # omit to run all six models
    offline=True,
    retune=True,
    with_backtest=True,
)
result["summary"]


## 5. Lectura de los resultados

`p_value_sign_test` es un contraste de signos de una cola sobre el número de
pliegues en que el modelo supera su suelo naïve, con H₀ = 0,5.

Es una **aproximación declarada**: bajo H₀ estricta la probabilidad de superar
el suelo en un pliegue no es exactamente 0,5 (depende de la tasa base del año y
de la discretización de la accuracy), y los pliegues comparten datos de
entrenamiento, así que no son independientes. Léase como indicador de
**estabilidad**, no como prueba formal de superioridad: ese papel corresponde al
binomial por pliegue de H1, que va en `walkforward_classification.csv`.


In [ ]:
summary = pd.read_csv(config.TABLES_DIR / "walkforward_summary.csv")
summary.sort_values("share_folds_above", ascending=False).head(15)


## 6. Figuras

Se reconstruyen desde las tablas persistidas, sin reentrenar nada.


In [ ]:
from tfm import evaluation

evaluation.build_walkforward_figures()
